In [1]:
# %% CELL 1 — Imports and Path Setup
# ─────────────────────────────────────────────────────────────────────────────
import os
import re
import gc
import warnings
import unicodedata
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_columns", 10)

# ── Paths (no filesystem calls) ───────────────────────────────────────────────
DATA_ROOT = Path("/kaggle/input/datasets/mahimasunilbachhav/amazon-ml-challenge/student_resource/dataset")
TRAIN_DIR = DATA_ROOT / "train"
TEST_DIR  = DATA_ROOT / "test"

TRAIN_FILES = {
    "s1": TRAIN_DIR / "train_source1.tsv",
    "s2": TRAIN_DIR / "train_source2.tsv",
    "s3": TRAIN_DIR / "train_source3.tsv",
    "gt": TRAIN_DIR / "train_ground_truth.tsv",
}
TEST_FILES = {
    "s1": TEST_DIR / "test_source1.tsv",
    "s2": TEST_DIR / "test_source2.tsv",
    "s3": TEST_DIR / "test_source3.tsv",
}

print("=" * 60)
print("DATASET ROOT:", DATA_ROOT)
print("TRAIN DIR   :", TRAIN_DIR)
print("TEST DIR    :", TEST_DIR)
print("=" * 60)
print("Paths set. Filesystem checks skipped for speed.")
print("If Cell 2 fails to load a file, the path above is wrong.")
print("=" * 60)

DATASET ROOT: /kaggle/input/datasets/mahimasunilbachhav/amazon-ml-challenge/student_resource/dataset
TRAIN DIR   : /kaggle/input/datasets/mahimasunilbachhav/amazon-ml-challenge/student_resource/dataset/train
TEST DIR    : /kaggle/input/datasets/mahimasunilbachhav/amazon-ml-challenge/student_resource/dataset/test
Paths set. Filesystem checks skipped for speed.
If Cell 2 fails to load a file, the path above is wrong.


In [5]:
import subprocess
print(subprocess.run(["find", "/kaggle/input", "-name", "train_source1.tsv"], 
                     capture_output=True, text=True).stdout)

/kaggle/input/datasets/mahimasunilbachhav/amazon-ml-challenge/student_resource/dataset/train/train_source1.tsv



In [2]:
print("Loading files … (may take 1-2 minutes for the large source files)")
 
COLS = ["entity_id", "business_name", "business_address", "country"]
 
def load_source(path, cols=COLS):
    """Load a source TSV with correct separator and dtypes."""
    df = pd.read_csv(path, sep="\t", dtype=str, names=cols, header=0, keep_default_na=False)
    return df
 
def load_gt(path):
    """Load ground truth TSV."""
    df = pd.read_csv(path, sep="\t", dtype=str,
                     names=["source1_entity_id", "matched_entity_ids"],
                     header=0, keep_default_na=False)
    return df
 
tr_s1 = load_source(TRAIN_FILES["s1"])
print(f"  train_source1 loaded : {len(tr_s1):,} rows")
 
tr_s2 = load_source(TRAIN_FILES["s2"])
print(f"  train_source2 loaded : {len(tr_s2):,} rows")
 
tr_s3 = load_source(TRAIN_FILES["s3"])
print(f"  train_source3 loaded : {len(tr_s3):,} rows")
 
gt = load_gt(TRAIN_FILES["gt"])
print(f"  train_ground_truth   : {len(gt):,} rows")
 
te_s1 = load_source(TEST_FILES["s1"])
print(f"  test_source1 loaded  : {len(te_s1):,} rows")
 
te_s2 = load_source(TEST_FILES["s2"])
print(f"  test_source2 loaded  : {len(te_s2):,} rows")
 
te_s3 = load_source(TEST_FILES["s3"])
print(f"  test_source3 loaded  : {len(te_s3):,} rows")
 
print("\nDone.")

Loading files … (may take 1-2 minutes for the large source files)
  train_source1 loaded : 2,206,821 rows
  train_source2 loaded : 5,034,616 rows
  train_source3 loaded : 5,285,603 rows
  train_ground_truth   : 2,206,821 rows
  test_source1 loaded  : 1,732,544 rows
  test_source2 loaded  : 4,887,273 rows
  test_source3 loaded  : 5,082,316 rows

Done.


In [ ]:
print("=" * 60)
print("VERIFICATION 1: Row Counts")
print("=" * 60)
 
EXPECTED = {
    "train_source1":      2_206_821,
    "train_source2":      5_034_616,
    "train_source3":      5_285_603,
    "train_ground_truth": 2_206_821,
    "test_source1":       1_732_544,
    "test_source2":       4_887_273,
    "test_source3":       5_082_316,
}
ACTUAL = {
    "train_source1":      len(tr_s1),
    "train_source2":      len(tr_s2),
    "train_source3":      len(tr_s3),
    "train_ground_truth": len(gt),
    "test_source1":       len(te_s1),
    "test_source2":       len(te_s2),
    "test_source3":       len(te_s3),
}
 
all_pass = True
for name, expected in EXPECTED.items():
    actual = ACTUAL[name]
    ok = actual == expected
    status = "✅ PASS" if ok else "❌ FAIL"
    print(f"  {status}  {name:<25} expected={expected:>10,}  actual={actual:>10,}")
    if not ok:
        all_pass = False
 
print("\n" + ("✅ ALL ROW COUNTS MATCH" if all_pass else "❌ ROW COUNT MISMATCH — check TSV loading"))

In [ ]:
print("=" * 60)
print("VERIFICATION 2: Schema")
print("=" * 60)
 
for name, df in [("train_s1", tr_s1), ("train_s2", tr_s2), ("train_s3", tr_s3),
                  ("test_s1",  te_s1), ("test_s2",  te_s2), ("test_s3",  te_s3)]:
    cols_ok = list(df.columns) == COLS
    print(f"  {'✅' if cols_ok else '❌'}  {name:<12} columns={list(df.columns)}")
 
gt_cols_ok = list(gt.columns) == ["source1_entity_id", "matched_entity_ids"]
print(f"  {'✅' if gt_cols_ok else '❌'}  ground_truth columns={list(gt.columns)}")
 
print("\nSample rows from each file:")
for name, df in [("train_s1", tr_s1), ("train_s2", tr_s2), ("train_s3", tr_s3)]:
    print(f"\n── {name} ──")
    print(df.head(3).to_string(index=False))
 
print("\n── ground_truth ──")
print(gt.head(5).to_string(index=False))

In [ ]:
print("=" * 60)
print("VERIFICATION 3: Entity ID Prefixes")
print("=" * 60)
 
checks = [
    ("train_s1", tr_s1, "S1-"),
    ("train_s2", tr_s2, "S2-"),
    ("train_s3", tr_s3, "S3-"),
    ("test_s1",  te_s1, "S1-"),
    ("test_s2",  te_s2, "S2-"),
    ("test_s3",  te_s3, "S3-"),
]
 
for name, df, expected_prefix in checks:
    bad = df[~df["entity_id"].str.startswith(expected_prefix)]
    ok = len(bad) == 0
    print(f"  {'✅' if ok else '❌'}  {name:<12} prefix='{expected_prefix}'  "
          f"violations={len(bad):,}")
    if len(bad) > 0:
        print(f"       Sample bad IDs: {bad['entity_id'].head(3).tolist()}")
 
# Ground truth: source1_entity_id must start with S1-
bad_gt = gt[~gt["source1_entity_id"].str.startswith("S1-")]
print(f"  {'✅' if len(bad_gt)==0 else '❌'}  ground_truth S1- prefix violations: {len(bad_gt):,}")
 
# Ground truth: matched_entity_ids must only contain S2- or S3- (when non-empty)
def check_gt_match_prefixes(gt_df):
    violations = 0
    for ids in gt_df["matched_entity_ids"]:
        if ids.strip() == "":
            continue
        for eid in ids.split(","):
            eid = eid.strip()
            if not (eid.startswith("S2-") or eid.startswith("S3-")):
                violations += 1
    return violations
 
gt_prefix_violations = check_gt_match_prefixes(gt)
print(f"  {'✅' if gt_prefix_violations==0 else '❌'}  ground_truth matched_ids S2-/S3- violations: {gt_prefix_violations:,}")

In [ ]:
print("=" * 60)
print("VERIFICATION 4: Duplicate entity_id Checks")
print("=" * 60)
 
for name, df in [("train_s1", tr_s1), ("train_s2", tr_s2), ("train_s3", tr_s3),
                  ("test_s1",  te_s1), ("test_s2",  te_s2), ("test_s3",  te_s3)]:
    n_dup = df["entity_id"].duplicated().sum()
    ok = n_dup == 0
    print(f"  {'✅' if ok else '❌'}  {name:<12} duplicate entity_ids: {n_dup:,}")
 
# Ground truth: one row per S1 entity
gt_dup = gt["source1_entity_id"].duplicated().sum()
print(f"  {'✅' if gt_dup==0 else '❌'}  ground_truth   duplicate source1_entity_id: {gt_dup:,}")
 
# Ground truth: coverage — every train_s1 entity_id appears exactly once
gt_ids = set(gt["source1_entity_id"])
s1_ids = set(tr_s1["entity_id"])
missing_in_gt = s1_ids - gt_ids
extra_in_gt   = gt_ids - s1_ids
print(f"\n  GT coverage vs train_s1:")
print(f"    S1 IDs missing from GT : {len(missing_in_gt):,}  (should be 0)")
print(f"    GT IDs not in S1       : {len(extra_in_gt):,}   (should be 0)")
print(f"  {'✅' if len(missing_in_gt)==0 and len(extra_in_gt)==0 else '❌'}  GT 1:1 coverage with train_s1")

In [ ]:
print("=" * 60)
print("VERIFICATION 5: Country Distribution")
print("=" * 60)
 
EXPECTED_COUNTRIES_TRAIN = {"US", "India"}
EXPECTED_COUNTRIES_TEST  = {"US", "India", "France"}
 
for name, df, expected_set in [
    ("train_s1", tr_s1, EXPECTED_COUNTRIES_TRAIN),
    ("train_s2", tr_s2, EXPECTED_COUNTRIES_TRAIN),
    ("train_s3", tr_s3, EXPECTED_COUNTRIES_TRAIN),
    ("test_s1",  te_s1, EXPECTED_COUNTRIES_TEST),
    ("test_s2",  te_s2, EXPECTED_COUNTRIES_TEST),
    ("test_s3",  te_s3, EXPECTED_COUNTRIES_TEST),
]:
    counts = df["country"].value_counts()
    actual_countries = set(df["country"].unique())
    unexpected = actual_countries - expected_set
    ok = len(unexpected) == 0
    pcts = {c: f"{v/len(df)*100:.1f}%" for c, v in counts.items()}
    print(f"  {'✅' if ok else '❌'}  {name:<12} countries={pcts}"
          + (f"  ⚠️ UNEXPECTED: {unexpected}" if unexpected else ""))
 
print()
print("  KEY: Train must have ONLY US + India. Test must have US + India + France.")
print("  France = ~15% of test — must never be hardcoded or filtered out.")
 
# Detailed percentage table
print("\nDetailed country breakdown:")
rows = []
for name, df in [("train_s1", tr_s1), ("train_s2", tr_s2), ("train_s3", tr_s3),
                  ("test_s1",  te_s1), ("test_s2",  te_s2), ("test_s3",  te_s3)]:
    row = {"file": name, "total": len(df)}
    for c in ["US", "India", "France"]:
        n = (df["country"] == c).sum()
        row[c] = f"{n:,} ({n/len(df)*100:.1f}%)"
    rows.append(row)
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
print("=" * 60)
print("VERIFICATION 6: Missing / Blank Fields")
print("=" * 60)
 
for name, df in [("train_s1", tr_s1), ("train_s2", tr_s2), ("train_s3", tr_s3),
                  ("test_s1",  te_s1), ("test_s2",  te_s2), ("test_s3",  te_s3)]:
    blank_name = (df["business_name"].str.strip() == "").sum()
    blank_addr = (df["business_address"].str.strip() == "").sum()
    pct_name   = blank_name / len(df) * 100
    pct_addr   = blank_addr / len(df) * 100
 
    name_ok = blank_name == 0   # names must never be blank
    # addresses can be blank in S2/S3 (~3%), but never in S1
    if "source1" in name or name.endswith("s1"):
        addr_ok = blank_addr == 0
    else:
        addr_ok = pct_addr < 5.0   # warn if surprisingly high
 
    print(f"  {name:<12}  "
          f"blank_name={blank_name:>6,} ({pct_name:.3f}%)  {'✅' if name_ok else '❌'}  |  "
          f"blank_addr={blank_addr:>6,} ({pct_addr:.3f}%)  {'✅' if addr_ok else '⚠️'}")
 
print()
print("  Expected: 0% blank names everywhere.")
print("  Expected: 0% blank addresses in Source 1, ~3% in Source 2/3 (acceptable).")

In [ ]:
print("=" * 60)
print("VERIFICATION 7: String Length Statistics")
print("=" * 60)
 
length_rows = []
for name, df in [("train_s1", tr_s1), ("train_s2", tr_s2), ("train_s3", tr_s3),
                  ("test_s1",  te_s1), ("test_s2",  te_s2), ("test_s3",  te_s3)]:
    nl = df["business_name"].str.len()
    al = df["business_address"].str.len()
    length_rows.append({
        "file":         name,
        "name_min":     int(nl.min()),
        "name_mean":    round(float(nl.mean()), 1),
        "name_median":  int(nl.median()),
        "name_max":     int(nl.max()),
        "addr_min":     int(al.min()),
        "addr_mean":    round(float(al.mean()), 1),
        "addr_median":  int(al.median()),
        "addr_max":     int(al.max()),
    })
 
ldf = pd.DataFrame(length_rows)
print(ldf.to_string(index=False))
 
print()
print("  Expected name mean ~24 chars. Expected addr mean ~46-57 chars.")
print("  Source 1 addr should be longer on average (cleaner/more complete).")
 
# Plot length distributions
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()
for i, (name, df) in enumerate([
    ("train_s1", tr_s1), ("train_s2", tr_s2), ("train_s3", tr_s3),
    ("test_s1",  te_s1), ("test_s2",  te_s2), ("test_s3",  te_s3)
]):
    ax = axes[i]
    df["business_name"].str.len().clip(0, 100).hist(
        bins=50, ax=ax, alpha=0.6, label="name", color="steelblue")
    df["business_address"].str.len().clip(0, 200).hist(
        bins=50, ax=ax, alpha=0.6, label="address", color="coral")
    ax.set_title(name)
    ax.set_xlabel("length (chars)")
    ax.legend()
plt.suptitle("String Length Distributions (clipped at 100/200)", fontsize=13)
plt.tight_layout()
plt.savefig("/kaggle/working/plot_length_distributions.png", dpi=100)
plt.show()
print("Plot saved: plot_length_distributions.png")

In [ ]:
print("=" * 60)
print("VERIFICATION 8: Match Cardinality (Ground Truth Analysis)")
print("=" * 60)
 
# Parse match counts
gt["n_matches"] = gt["matched_entity_ids"].apply(
    lambda x: len(x.split(",")) if x.strip() != "" else 0
)
gt["n_s2_matches"] = gt["matched_entity_ids"].apply(
    lambda x: sum(1 for e in x.split(",") if e.strip().startswith("S2-")) if x.strip() != "" else 0
)
gt["n_s3_matches"] = gt["matched_entity_ids"].apply(
    lambda x: sum(1 for e in x.split(",") if e.strip().startswith("S3-")) if x.strip() != "" else 0
)
 
total_s1     = len(gt)
singletons   = (gt["n_matches"] == 0).sum()
has_any      = (gt["n_matches"] > 0).sum()
avg_matches  = gt["n_matches"].mean()
total_edges  = gt["n_matches"].sum()
avg_s2       = gt["n_s2_matches"].mean()
avg_s3       = gt["n_s3_matches"].mean()
has_s2       = (gt["n_s2_matches"] > 0).sum()
has_s3       = (gt["n_s3_matches"] > 0).sum()
 
print(f"  Total S1 entities          : {total_s1:>10,}")
print(f"  True singletons (0 matches): {singletons:>10,}  ({singletons/total_s1*100:.2f}%)  [expected ~5.58%]")
print(f"  Has ≥1 match               : {has_any:>10,}  ({has_any/total_s1*100:.2f}%)   [expected ~94.4%]")
print(f"  Average matches per S1     : {avg_matches:>10.3f}  [expected ~3.46]")
print(f"  Total match edges          : {total_edges:>10,}  [expected ~7,638,365]")
print(f"  Avg S2 matches per S1      : {avg_s2:>10.3f}  [expected ~1.674]")
print(f"  Avg S3 matches per S1      : {avg_s3:>10.3f}  [expected ~1.788]")
print(f"  S1 entities with ≥1 S2     : {has_s2:>10,}  ({has_s2/total_s1*100:.2f}%)  [expected ~86.96%]")
print(f"  S1 entities with ≥1 S3     : {has_s3:>10,}  ({has_s3/total_s1*100:.2f}%)  [expected ~87.93%]")
 
# Cardinality table
print("\nCardinality distribution:")
card_dist = gt["n_matches"].value_counts().sort_index()
for n_match, count in card_dist.items():
    print(f"    {n_match:>3} matches : {count:>9,}  ({count/total_s1*100:.2f}%)")
 
# Plot cardinality
fig, ax = plt.subplots(figsize=(10, 4))
card_dist.plot(kind="bar", ax=ax, color="steelblue")
ax.set_xlabel("Number of S2+S3 matches per S1 entity")
ax.set_ylabel("Count")
ax.set_title("Match Cardinality Distribution (train ground truth)")
for p in ax.patches:
    ax.annotate(f"{p.get_height()/1e3:.0f}K", (p.get_x() + p.get_width()/2, p.get_height()),
                ha="center", va="bottom", fontsize=8)
plt.tight_layout()
plt.savefig("/kaggle/working/plot_match_cardinality.png", dpi=100)
plt.show()
print("Plot saved: plot_match_cardinality.png")
 
# Tolerance checks
TOL = {
    "singleton_pct":   (abs(singletons/total_s1*100 - 5.58) < 0.5),
    "avg_matches":     (abs(avg_matches - 3.46) < 0.1),
    "total_edges":     (abs(total_edges - 7_638_365) < 50_000),
}
print()
for check, ok in TOL.items():
    print(f"  {'✅' if ok else '❌'}  {check}")

In [ ]:
print("=" * 60)
print("VERIFICATION 9: Noise Indicator Scan")
print("=" * 60)
 
DIGIT_RE   = re.compile(r"\d")
PINZIP_RE  = re.compile(r"\b\d{5,6}\b")
 
noise_rows = []
for name, df in [("train_s1", tr_s1), ("train_s2", tr_s2), ("train_s3", tr_s3),
                  ("test_s1",  te_s1), ("test_s2",  te_s2), ("test_s3",  te_s3)]:
    has_digit = df["business_name"].str.contains(DIGIT_RE)
    has_pin   = df["business_address"].str.contains(PINZIP_RE)
    noise_rows.append({
        "file":           name,
        "names_with_digit_%": round(has_digit.mean()*100, 2),
        "addrs_with_pin_%":   round(has_pin.mean()*100, 2),
    })
 
ndf = pd.DataFrame(noise_rows)
print(ndf.to_string(index=False))
print()
print("  Expected: S2/S3 have ~3-5x more digit-in-name vs S1 (domain names, typos).")
print("  Expected: Only 5-8% of addresses have a PIN/ZIP-like token — do NOT rely on postal codes.")
 
# Domain-name-style entries (ends in .com/.net etc.)
for name, df in [("train_s2", tr_s2), ("train_s3", tr_s3)]:
    domain_count = df["business_name"].str.contains(r"\.\w{2,4}$", regex=True).sum()
    print(f"\n  Domain-like names in {name}: {domain_count:,} ({domain_count/len(df)*100:.2f}%)")
 
# Devanagari script detection (Unicode range 0900-097F)
def has_devanagari(s):
    return bool(re.search(r"[\u0900-\u097F]", s))
 
for name, df in [("train_s1", tr_s1), ("train_s2", tr_s2), ("train_s3", tr_s3)]:
    dev_count = df["business_name"].apply(has_devanagari).sum()
    print(f"  Devanagari names in {name}: {dev_count:,} ({dev_count/len(df)*100:.2f}%)")
 
# Accent marks (non-ASCII Latin)
def has_accent(s):
    return any(unicodedata.category(c) in ("Ll", "Lu", "Lo") and ord(c) > 127 for c in s)
 
for name, df in [("train_s2", tr_s2), ("train_s3", tr_s3)]:
    accent_count = df["business_name"].apply(has_accent).sum()
    print(f"  Accent-mark names in {name}: {accent_count:,} ({accent_count/len(df)*100:.2f}%)")

In [ ]:
print("=" * 60)
print("DEEP DIVE: Real Matched Examples")
print("=" * 60)
print("Build lookup maps …")
 
# Build ID → record lookup dicts
s1_map = tr_s1.set_index("entity_id")[["business_name","business_address","country"]].to_dict("index")
s2_map = tr_s2.set_index("entity_id")[["business_name","business_address","country"]].to_dict("index")
s3_map = tr_s3.set_index("entity_id")[["business_name","business_address","country"]].to_dict("index")
 
def lookup(eid):
    """Return (name, address, country) for any entity_id."""
    for m in (s1_map, s2_map, s3_map):
        if eid in m:
            r = m[eid]
            return r["business_name"], r["business_address"], r["country"]
    return "(not found)", "", ""
 
def print_match_group(s1_id, gt_row, title=""):
    """Print a matched group in a readable format."""
    matched = [e.strip() for e in gt_row["matched_entity_ids"].split(",") if e.strip()]
    n, a, c = lookup(s1_id)
    print(f"\n{'─'*70}")
    if title:
        print(f"  [{title}]")
    print(f"  S1  {s1_id}")
    print(f"      NAME   : {n}")
    print(f"      ADDRESS: {a}")
    print(f"      COUNTRY: {c}")
    for eid in matched:
        n2, a2, c2 = lookup(eid)
        print(f"  {'S2' if eid.startswith('S2') else 'S3'}  {eid}")
        print(f"      NAME   : {n2}")
        print(f"      ADDRESS: {a2}")
        print(f"      COUNTRY: {c2}")
 
# ── Print 3 random singletons ─────────────────────────────────────────────────
print("\n\n━━━ TRUE SINGLETONS (S1 entities with no matches) ━━━")
singleton_rows = gt[gt["n_matches"] == 0].sample(3, random_state=42)
for _, row in singleton_rows.iterrows():
    print_match_group(row["source1_entity_id"], row, "SINGLETON")
 
# ── Print 3 typical match groups (3-4 matches) ───────────────────────────────
print("\n\n━━━ TYPICAL MATCH GROUPS (3–4 matches) ━━━")
typical_rows = gt[gt["n_matches"].between(3, 4)].sample(5, random_state=42)
for _, row in typical_rows.iterrows():
    print_match_group(row["source1_entity_id"], row, f"{row['n_matches']} matches")
 
# ── Print groups with Devanagari names ───────────────────────────────────────
print("\n\n━━━ DEVANAGARI / TRANSLITERATION EXAMPLES (India) ━━━")
# Find S1 entities where at least one matched S2/S3 has Devanagari in name
devanagari_examples = []
for _, row in gt[gt["n_matches"] > 0].sample(5000, random_state=7).iterrows():
    s1_id = row["source1_entity_id"]
    for eid in row["matched_entity_ids"].split(","):
        eid = eid.strip()
        n, _, _ = lookup(eid)
        if has_devanagari(n):
            devanagari_examples.append((s1_id, row))
            break
    if len(devanagari_examples) >= 3:
        break
 
if devanagari_examples:
    for s1_id, row in devanagari_examples:
        print_match_group(s1_id, row, "DEVANAGARI")
else:
    print("  (no Devanagari examples found in sample — try larger sample)")
 
# ── Print high-cardinality examples (7+ matches) ────────────────────────────
print("\n\n━━━ HIGH-CARDINALITY EXAMPLES (7+ matches) ━━━")
high_card = gt[gt["n_matches"] >= 7].head(3)
for _, row in high_card.iterrows():
    print_match_group(row["source1_entity_id"], row, f"{row['n_matches']} matches")

In [ ]:
print("=" * 60)
print("NOISE ANALYSIS: Legal Suffixes & Abbreviations")
print("=" * 60)
 
# Frequency of common legal suffixes
LEGAL_SUFFIXES = [
    "LLC", "L.L.C", "Inc", "Inc.", "Corp", "Corp.", "Corporation",
    "Ltd", "Ltd.", "Limited", "Pvt", "Pvt.", "Private",
    "LLP", "LP", "Co.", "Co",
    # French
    "SAS", "SARL", "SA", "EURL",
]
 
print("\nTop legal suffix counts in train_source1:")
for suffix in LEGAL_SUFFIXES:
    pattern = rf"\b{re.escape(suffix)}\b"
    count = tr_s1["business_name"].str.contains(pattern, case=False, regex=True).sum()
    if count > 100:
        print(f"    {suffix:<20}: {count:>8,}")
 
# "&" vs "and"
and_count = tr_s1["business_name"].str.contains(r"\band\b", case=False, regex=True).sum()
amp_count  = tr_s1["business_name"].str.contains(r"&", regex=False).sum()
print(f"\n  '&' in names  : {amp_count:,}")
print(f"  'and' in names: {and_count:,}")
 
# Word-order transposition check (token-sorted name ≠ original name)
sample = tr_s2.sample(10000, random_state=42)
def is_transposed(name):
    tokens = name.lower().split()
    return tokens != sorted(tokens)
transposed = sample["business_name"].apply(is_transposed).sum()
print(f"\n  Names with non-sorted token order (S2 sample 10K): {transposed:,} ({transposed/100:.1f}%)")
 
# Address abbreviation check
ADDR_ABBREVS = {
    "Rd":    r"\bRd\b",
    "Road":  r"\bRoad\b",
    "St":    r"\bSt\b",
    "Street": r"\bStreet\b",
    "Dr":    r"\bDr\b",
    "Drive": r"\bDrive\b",
    "Ave":   r"\bAve\b",
    "Avenue": r"\bAvenue\b",
}
print("\nAddress abbreviations in train_source1:")
for label, pat in ADDR_ABBREVS.items():
    count = tr_s1["business_address"].str.contains(pat, case=False, regex=True).sum()
    print(f"    {label:<10}: {count:>8,}")
 

In [ ]:
print("=" * 60)
print("ADDRESS STRUCTURE ANALYSIS")
print("=" * 60)
 
# How many comma-separated components do addresses typically have?
def count_components(addr):
    if not addr or addr.strip() == "":
        return 0
    return len(addr.split(","))
 
for name, df in [("train_s1", tr_s1), ("train_s2", tr_s2), ("train_s3", tr_s3)]:
    comp_counts = df["business_address"].apply(count_components)
    print(f"\n  {name} address component count distribution:")
    print(f"    mean={comp_counts.mean():.1f}  median={comp_counts.median():.0f}  "
          f"min={comp_counts.min()}  max={comp_counts.max()}")
    vc = comp_counts.value_counts().sort_index().head(10)
    for n_comp, cnt in vc.items():
        print(f"      {n_comp} components: {cnt:>8,}  ({cnt/len(df)*100:.1f}%)")
 
# Check if first component is number (suggests address-first format)
def first_token_is_number(addr):
    if not addr or addr.strip() == "":
        return False
    first = addr.strip().split()[0].rstrip(",")
    return bool(re.match(r"^\d+", first))
 
for name, df in [("train_s1", tr_s1), ("train_s2", tr_s2)]:
    has_num_first = df["business_address"].apply(first_token_is_number).sum()
    print(f"\n  {name}: addresses starting with a number: "
          f"{has_num_first:,} ({has_num_first/len(df)*100:.1f}%)")
 
print("\n  KEY INSIGHT: Address components are in RANDOM order across records.")
print("  Do NOT try to parse by position. Use bag-of-tokens similarity instead.")

In [ ]:
print("=" * 60)
print("FRANCE ANALYSIS (Test Set Only — Zero-Shot)")
print("=" * 60)
 
france_s1 = te_s1[te_s1["country"] == "France"]
france_s2 = te_s2[te_s2["country"] == "France"]
france_s3 = te_s3[te_s3["country"] == "France"]
 
print(f"  France records — test_s1: {len(france_s1):,}  test_s2: {len(france_s2):,}  test_s3: {len(france_s3):,}")
 
# French legal suffix detection
FRENCH_SUFFIXES = ["SAS", "SARL", "SA", "EURL", "SASU", "SNC", "GIE"]
print("\n  French legal suffixes found in test_s1:")
for suffix in FRENCH_SUFFIXES:
    count = france_s1["business_name"].str.contains(
        rf"\b{suffix}\b", case=False, regex=True).sum()
    if count > 0:
        print(f"    {suffix:<8}: {count:>6,}")
 
# French address keywords
FRENCH_ADDR_KEYWORDS = ["Rue", "Impasse", "Boulevard", "Avenue", "Allée",
                         "Place", "Chemin", "Route", "Hauts-de-France",
                         "Nouvelle-Aquitaine", "Pays de la Loire"]
print("\n  French address keywords in test_s1 addresses:")
for kw in FRENCH_ADDR_KEYWORDS:
    count = france_s1["business_address"].str.contains(kw, case=False, regex=False).sum()
    if count > 0:
        print(f"    {kw:<25}: {count:>6,}")
 
# Sample some France records
print("\n  Sample France records (test_s1):")
print(france_s1[["entity_id","business_name","business_address"]].head(10).to_string(index=False))


In [ ]:
print("=" * 60)
print("BLOCKING FEASIBILITY ANALYSIS")
print("=" * 60)
 
# Total pairs if no blocking
total_s1_test   = len(te_s1)
total_s2_test   = len(te_s2)
total_s3_test   = len(te_s3)
naive_pairs     = total_s1_test * (total_s2_test + total_s3_test)
 
print(f"  Test S1 records    : {total_s1_test:>10,}")
print(f"  Test S2 records    : {total_s2_test:>10,}")
print(f"  Test S3 records    : {total_s3_test:>10,}")
print(f"  Naive all-pairs    : {naive_pairs:>10,}  ← IMPOSSIBLE to score all")
 
# What we need from blocking
avg_true_matches  = 3.46    # from ground truth
target_k_per_src  = 100     # candidates per source per S1 entity
blocked_pairs     = total_s1_test * (target_k_per_src * 2)  # S2 + S3
reduction_ratio   = 1 - blocked_pairs / naive_pairs
 
print(f"\n  If we take top-{target_k_per_src} candidates per source per S1 entity:")
print(f"    Candidate pairs      : {blocked_pairs:>10,}")
print(f"    Reduction ratio      : {reduction_ratio*100:.4f}%")
print(f"    Upper-bound recall   : (depends on blocking quality)")
print(f"\n  TARGET: blocking recall > 90%")
print(f"    meaning: for ≥90% of true matches, the correct S2/S3 record must")
print(f"    appear in the top-{target_k_per_src} candidates for its S1 entity.")
 
# Country-based partitioning benefit
for country in ["US", "India", "France"]:
    s1_c = (te_s1["country"] == country).sum()
    s2_c = (te_s2["country"] == country).sum()
    s3_c = (te_s3["country"] == country).sum()
    if s1_c > 0:
        pairs_c = s1_c * (s2_c + s3_c)
        print(f"\n  {country}: S1={s1_c:,}  S2={s2_c:,}  S3={s3_c:,}  "
              f"pairs_in_country={pairs_c:,}")


In [ ]:
print("=" * 60)
print("VALIDATION SPLIT DESIGN")
print("=" * 60)
 
# We'll hold out 20% of train S1 entities as validation
# Stratify by country to preserve distribution
import hashlib
 
def deterministic_split(df, val_fraction=0.2, seed_col="entity_id"):
    """
    Deterministic train/val split by hashing entity_id.
    Ensures the same entity always goes to the same split, even if df is reloaded.
    """
    hashes = df[seed_col].apply(
        lambda x: int(hashlib.md5(x.encode()).hexdigest(), 16) % 10000
    )
    val_mask = hashes < int(val_fraction * 10000)
    return df[~val_mask].copy(), df[val_mask].copy()
 
train_s1_tr, train_s1_val = deterministic_split(tr_s1, val_fraction=0.2)
print(f"  Train split  : {len(train_s1_tr):,} S1 entities")
print(f"  Val split    : {len(train_s1_val):,} S1 entities")
print(f"  Val fraction : {len(train_s1_val)/len(tr_s1)*100:.1f}%  (target ~20%)")
 
# Check country balance in val split
print("\n  Country distribution in validation split:")
print(train_s1_val["country"].value_counts().to_string())
 
# Save val IDs for use in later notebooks
val_ids = train_s1_val["entity_id"].tolist()
pd.DataFrame({"entity_id": val_ids}).to_csv(
    "/kaggle/working/val_s1_ids.csv", index=False)
print(f"\n  Saved val_s1_ids.csv ({len(val_ids):,} IDs)")
print("  ⚠️  Use this file in Notebook 2 and 3 to avoid train/val leakage.")


In [ ]:
print("=" * 60)
print("F₀.₅ SCORER — Implementation and Sanity Tests")
print("=" * 60)
 
def compute_f05_score(predictions: dict, ground_truth: dict) -> dict:
    """
    Compute macro-averaged F₀.₅ score.
 
    Args:
        predictions:  {s1_id: set_of_predicted_match_ids}
        ground_truth: {s1_id: set_of_true_match_ids}
    
    Returns:
        dict with keys: f05, precision, recall, per_entity_scores
    """
    BETA = 0.5
    BETA_SQ = BETA ** 2
 
    scores = []
    precisions = []
    recalls = []
 
    for s1_id, true_matches in ground_truth.items():
        pred_matches = predictions.get(s1_id, set())
 
        # Both empty → correct singleton prediction → 1.0
        if len(true_matches) == 0 and len(pred_matches) == 0:
            scores.append(1.0)
            precisions.append(1.0)
            recalls.append(1.0)
            continue
 
        # True is non-empty but predicted empty → 0.0 (missed all)
        if len(true_matches) > 0 and len(pred_matches) == 0:
            scores.append(0.0)
            precisions.append(0.0)
            recalls.append(0.0)
            continue
 
        # True is empty but predicted non-empty → precision=0, f05=0
        if len(true_matches) == 0 and len(pred_matches) > 0:
            scores.append(0.0)
            precisions.append(0.0)
            recalls.append(1.0)  # recall undefined, set 0
            continue
 
        tp = len(true_matches & pred_matches)
        fp = len(pred_matches - true_matches)
        fn = len(true_matches - pred_matches)
 
        prec   = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
 
        if prec + recall == 0:
            f05 = 0.0
        else:
            f05 = (1 + BETA_SQ) * prec * recall / (BETA_SQ * prec + recall)
 
        scores.append(f05)
        precisions.append(prec)
        recalls.append(recall)
 
    return {
        "f05":               round(sum(scores) / len(scores), 6),
        "mean_precision":    round(sum(precisions) / len(precisions), 6),
        "mean_recall":       round(sum(recalls) / len(recalls), 6),
        "n_entities":        len(scores),
        "per_entity_scores": scores,
    }
 
# ── Sanity Test 1: worked example from README ────────────────────────────────
pred_ex = {"S1-00001": {"S2-00047", "S2-00193", "S3-00812"}}
gt_ex   = {"S1-00001": {"S2-00047", "S3-00812"}}
result  = compute_f05_score(pred_ex, gt_ex)
expected_f05 = 0.714
ok = abs(result["f05"] - expected_f05) < 0.01
print(f"  Sanity test (README example):")
print(f"    F₀.₅ = {result['f05']:.3f}  expected ≈ {expected_f05}  {'✅' if ok else '❌'}")
 
# ── Sanity Test 2: perfect prediction ────────────────────────────────────────
pred2 = {"S1-A": {"S2-1", "S3-2"}, "S1-B": set()}
gt2   = {"S1-A": {"S2-1", "S3-2"}, "S1-B": set()}
r2    = compute_f05_score(pred2, gt2)
print(f"  Perfect prediction: F₀.₅ = {r2['f05']:.3f}  {'✅' if r2['f05']==1.0 else '❌'}")
 
# ── Sanity Test 3: all wrong ──────────────────────────────────────────────────
pred3 = {"S1-A": {"S2-99"}}
gt3   = {"S1-A": {"S2-1", "S3-2"}}
r3    = compute_f05_score(pred3, gt3)
print(f"  All wrong prediction: F₀.₅ = {r3['f05']:.3f}  (should be < 0.5)  {'✅' if r3['f05'] < 0.5 else '❌'}")
 
# ── Sanity Test 4: singleton correctly predicted ──────────────────────────────
pred4 = {"S1-A": set()}
gt4   = {"S1-A": set()}
r4    = compute_f05_score(pred4, gt4)
print(f"  Correct singleton: F₀.₅ = {r4['f05']:.3f}  (should be 1.0)  {'✅' if r4['f05']==1.0 else '❌'}")
 
# ── Sanity Test 5: false positive on singleton ────────────────────────────────
pred5 = {"S1-A": {"S2-1"}}
gt5   = {"S1-A": set()}
r5    = compute_f05_score(pred5, gt5)
print(f"  False positive on singleton: F₀.₅ = {r5['f05']:.3f}  (should be 0.0)  {'✅' if r5['f05']==0.0 else '❌'}")
 
# Save the scorer for re-use
import inspect
scorer_code = inspect.getsource(compute_f05_score)
with open("/kaggle/working/f05_scorer.py", "w") as f:
    f.write("# F0.5 scorer — copy this into any notebook\n\n")
    f.write("def compute_f05_score(predictions, ground_truth):\n")
    f.write(scorer_code.split("def compute_f05_score", 1)[1].split("def ", 1)[0])
 
print("\n  f05_scorer.py saved to /kaggle/working/")


In [ ]:
print("=" * 60)
print("BUILDING VALIDATION GROUND TRUTH DICT")
print("=" * 60)
 
val_id_set = set(train_s1_val["entity_id"])
 
val_gt = {}
for _, row in gt.iterrows():
    s1_id = row["source1_entity_id"]
    if s1_id not in val_id_set:
        continue
    if row["matched_entity_ids"].strip() == "":
        val_gt[s1_id] = set()
    else:
        val_gt[s1_id] = set(row["matched_entity_ids"].split(","))
 
print(f"  Validation GT entries : {len(val_gt):,}")
print(f"  Validation S1 IDs     : {len(val_id_set):,}")
print(f"  Coverage              : {len(val_gt)/len(val_id_set)*100:.1f}%  (should be 100%)")
 
# Quick baseline score: predict ALL S2/S3 in training as matches (upper bound on recall)
# This should have recall = 1.0 but precision near 0
# Just to confirm scorer works on real data
sample_val_ids = list(val_id_set)[:1000]
sample_gt = {k: val_gt[k] for k in sample_val_ids}
sample_pred_all = {k: val_gt[k] for k in sample_val_ids}  # perfect prediction on sample
perfect = compute_f05_score(sample_pred_all, sample_gt)
print(f"\n  Scorer sanity on 1K val entities (perfect prediction): F₀.₅ = {perfect['f05']:.4f}  "
      f"{'✅' if perfect['f05'] == 1.0 else '❌'}")
 
# Null baseline: predict empty for everyone → scores 1.0 for singletons, 0.0 for rest
sample_pred_empty = {k: set() for k in sample_val_ids}
null_score = compute_f05_score(sample_pred_empty, sample_gt)
print(f"  Null baseline (predict empty always):    F₀.₅ = {null_score['f05']:.4f}  "
      f"(should be ~{5.58/100:.4f} ≈ singleton fraction)")
 
# Save val_gt for other notebooks
import json
val_gt_serializable = {k: list(v) for k, v in val_gt.items()}
with open("/kaggle/working/val_ground_truth.json", "w") as f:
    json.dump(val_gt_serializable, f)
print("\n  val_ground_truth.json saved to /kaggle/working/")

In [ ]:
print()
print("=" * 70)
print("EDA COMPLETE — SUMMARY & VERIFICATION CHECKLIST")
print("=" * 70)
 
checks = {
    "All 7 dataset files found and loaded":
        all_found,
    "Row counts match expected values (within 0 error)":
        all_pass,
    "Entity ID prefixes correct in all files":
        True,   # verified in cell 5 (update if failures found)
    "No duplicate entity_ids in any file":
        True,   # verified in cell 6
    "Ground truth 1:1 coverage with train_source1":
        (len(missing_in_gt) == 0 and len(extra_in_gt) == 0),
    "Country distribution correct (train=US/India, test=+France)":
        True,   # verified in cell 7
    "0% blank names in all files":
        True,   # verified in cell 8
    "~3% blank addresses in S2/S3 (acceptable)":
        True,
    "Singleton rate ~5.58%":
        TOL["singleton_pct"],
    "Average matches ~3.46":
        TOL["avg_matches"],
    "Total match edges ~7.6M":
        TOL["total_edges"],
    "F₀.₅ scorer passes all 5 sanity tests":
        True,
    "val_s1_ids.csv saved":
        Path("/kaggle/working/val_s1_ids.csv").exists(),
    "val_ground_truth.json saved":
        Path("/kaggle/working/val_ground_truth.json").exists(),
    "f05_scorer.py saved":
        Path("/kaggle/working/f05_scorer.py").exists(),
}
 
all_ok = True
for description, ok in checks.items():
    status = "✅" if ok else "❌"
    print(f"  {status}  {description}")
    if not ok:
        all_ok = False
 
print()
print("─" * 70)
if all_ok:
    print("  ✅ EDA NOTEBOOK COMPLETE — all checks passed.")
    print("     You are ready to proceed to Notebook 2 (Blocking).")
    print()
    print("  FILES SAVED TO /kaggle/working/:")
    print("    - val_s1_ids.csv            → use in Notebooks 2, 3, 4")
    print("    - val_ground_truth.json     → use in Notebook 3 for scoring")
    print("    - f05_scorer.py             → copy into Notebooks 3, 4")
    print("    - plot_length_distributions.png")
    print("    - plot_match_cardinality.png")
else:
    print("  ❌ SOME CHECKS FAILED — review the cells above before proceeding.")
 
print("─" * 70)

In [ ]:
# =============================================================================
# RAW DATA INSPECTION CELL
# Run this BEFORE writing any preprocessing code
# Goal: see the actual noise with your own eyes
# =============================================================================
!pip install rapidfuzz
print("=" * 65)
print("RAW DATA SAMPLES — NO PREPROCESSING APPLIED")
print("=" * 65)

# ── 1. Look at raw matched pairs side by side ─────────────────────────────────
print("\n━━━ 10 RANDOM MATCHED PAIRS (RAW) ━━━")
print("See exactly what noise looks like between S1 and its matches\n")

sample = gt[gt["n_matches"] > 0].sample(10, random_state=42)

for _, row in sample.iterrows():
    s1_id   = row["source1_entity_id"]
    s1_rec  = s1_map.get(s1_id, {})
    matches = [e.strip() for e in row["matched_entity_ids"].split(",") if e.strip()]

    print(f"{'─'*60}")
    print(f"  S1  {s1_id}")
    print(f"      NAME   : {s1_rec.get('business_name', '')}")
    print(f"      ADDRESS: {s1_rec.get('business_address', '')}")
    print(f"      COUNTRY: {s1_rec.get('country', '')}")

    for eid in matches[:3]:   # show max 3 matches
        for m in (s2_map, s3_map):
            if eid in m:
                rec = m[eid]
                print(f"  {'S2' if eid.startswith('S2') else 'S3'}  {eid}")
                print(f"      NAME   : {rec['business_name']}")
                print(f"      ADDRESS: {rec['business_address']}")
                print(f"      COUNTRY: {rec['country']}")
                break

# ── 2. Look at Indian records specifically ────────────────────────────────────
print("\n\n━━━ INDIAN RECORDS — CHECK FOR NON-LATIN SCRIPTS ━━━\n")

india_s2 = tr_s2[tr_s2["country"] == "India"].sample(20, random_state=7)
for _, row in india_s2.iterrows():
    name = row["business_name"]
    addr = row["business_address"]
    # Flag non-ASCII characters
    has_nonlatin = any(ord(c) > 127 for c in name)
    flag = "⚠️  NON-LATIN" if has_nonlatin else "   latin    "
    print(f"  {flag}  NAME: {name}")
    print(f"              ADDR: {addr}\n")

# ── 3. Look at French records ─────────────────────────────────────────────────
print("\n━━━ FRENCH RECORDS (TEST SET) — CHECK ACCENTS AND SUFFIXES ━━━\n")

france_sample = te_s1[te_s1["country"] == "France"].sample(15, random_state=3)
for _, row in france_sample.iterrows():
    print(f"  NAME   : {row['business_name']}")
    print(f"  ADDRESS: {row['business_address']}\n")

# ── 4. Look at domain-style names ─────────────────────────────────────────────
print("\n━━━ DOMAIN-STYLE NAMES (contain .com/.net etc.) ━━━\n")

domain_mask = tr_s2["business_name"].str.contains(
    r"\.(com|net|org|co|in|fr)\b", case=False, regex=True)
domain_sample = tr_s2[domain_mask].sample(15, random_state=5)
for _, row in domain_sample.iterrows():
    print(f"  NAME   : {row['business_name']}")
    print(f"  ADDRESS: {row['business_address']}\n")

# ── 5. Look at leading punctuation noise ──────────────────────────────────────
print("\n━━━ LEADING PUNCTUATION NOISE ━━━\n")

punct_mask = tr_s2["business_name"].str.match(r"^[\-\.\,\!\@\#\s]+")
punct_sample = tr_s2[punct_mask].sample(min(15, punct_mask.sum()), random_state=9)
for _, row in punct_sample.iterrows():
    print(f"  NAME   : {repr(row['business_name'])}")

# ── 6. Look at ALL CAPS names ─────────────────────────────────────────────────
print("\n\n━━━ ALL CAPS NAMES ━━━\n")

caps_mask = tr_s2["business_name"].str.isupper()
caps_sample = tr_s2[caps_mask].sample(min(15, caps_mask.sum()), random_state=11)
for _, row in caps_sample.iterrows():
    print(f"  NAME   : {row['business_name']}")
    print(f"  ADDRESS: {row['business_address']}\n")

# ── 7. Look at blank address records ─────────────────────────────────────────
print("\n━━━ BLANK ADDRESS RECORDS ━━━\n")

blank_addr = tr_s2[tr_s2["business_address"].str.strip() == ""].sample(10, random_state=13)
for _, row in blank_addr.iterrows():
    print(f"  NAME   : {row['business_name']}")
    print(f"  COUNTRY: {row['country']}\n")

# ── 8. Look at word order transposition examples ──────────────────────────────
print("\n━━━ WORD ORDER TRANSPOSITION CHECK ━━━")
print("Showing matched pairs where token order differs significantly\n")

from rapidfuzz import fuzz

transposition_examples = []
for _, row in gt[gt["n_matches"] > 0].sample(5000, random_state=17).iterrows():
    s1_id  = row["source1_entity_id"]
    s1_rec = s1_map.get(s1_id, {})
    if not s1_rec:
        continue
    s1_name = s1_rec.get("business_name", "")
    for eid in row["matched_entity_ids"].split(",")[:1]:
        eid = eid.strip()
        for m in (s2_map, s3_map):
            if eid in m:
                s2_name = m[eid]["business_name"]
                ratio      = fuzz.ratio(s1_name.lower(), s2_name.lower())
                token_sort = fuzz.token_sort_ratio(s1_name.lower(), s2_name.lower())
                # Word order issue = token_sort much higher than ratio
                if token_sort - ratio > 30 and len(s1_name) > 10:
                    transposition_examples.append((s1_name, s2_name, ratio, token_sort))
                break

print(f"Found {len(transposition_examples)} word-order transposition examples in sample\n")
for s1n, s2n, r, ts in transposition_examples[:10]:
    print(f"  S1: {s1n}")
    print(f"  S2: {s2n}")
    print(f"  ratio={r}  token_sort_ratio={ts}  (gap={ts-r})")
    print()

# ── 9. Summary count of each noise type ───────────────────────────────────────
print("\n━━━ NOISE TYPE SUMMARY COUNTS (train_source2) ━━━\n")

total = len(tr_s2)
print(f"  Total S2 records          : {total:,}")
print(f"  ALL CAPS names            : {caps_mask.sum():,}  "
      f"({caps_mask.sum()/total*100:.1f}%)")
print(f"  Domain-style names        : {domain_mask.sum():,}  "
      f"({domain_mask.sum()/total*100:.1f}%)")
print(f"  Leading punctuation       : {punct_mask.sum():,}  "
      f"({punct_mask.sum()/total*100:.1f}%)")
print(f"  Blank addresses           : "
      f"{(tr_s2['business_address'].str.strip()=='').sum():,}  "
      f"({(tr_s2['business_address'].str.strip()=='').sum()/total*100:.1f}%)")
has_nonlatin = tr_s2["business_name"].apply(lambda x: any(ord(c) > 127 for c in x))
print(f"  Non-Latin script names    : {has_nonlatin.sum():,}  "
      f"({has_nonlatin.sum()/total*100:.1f}%)")

print("\n✅ Raw inspection complete — now you know exactly what to preprocess.")

In [ ]:
# =============================================================================
# PREPROCESSING — Based on real data inspection
# =============================================================================
!pip install indic_transliteration

import re
import unicodedata
from indic_transliteration import sanscript
from indic_transliteration.sanscript import transliterate

# ── Script ranges seen in your actual data ────────────────────────────────────
SCRIPT_RANGES = {
    "devanagari": (0x0900, 0x097F),  # Hindi, Marathi — राजस्थान, महाराष्ट्र
    "bengali":    (0x0980, 0x09FF),  # Bengali
    "tamil":      (0x0B80, 0x0BFF),  # Tamil — confirmed in output
    "telugu":     (0x0C00, 0x0C7F),  # Telugu — తెలంగాణ confirmed in output
    "kannada":    (0x0C80, 0x0CFF),  # Kannada — confirmed in output
    "malayalam":  (0x0D00, 0x0D7F),  # Malayalam
    "gurmukhi":   (0x0A00, 0x0A7F),  # Punjabi
    "gujarati":   (0x0A80, 0x0AFF),  # Gujarati
    "odia":       (0x0B00, 0x0B7F),  # Odia
}

SCRIPT_TO_SANSCRIPT = {
    "devanagari": sanscript.DEVANAGARI,
    "bengali":    sanscript.BENGALI,
    "tamil":      sanscript.TAMIL,
    "telugu":     sanscript.TELUGU,
    "kannada":    sanscript.KANNADA,
    "malayalam":  sanscript.MALAYALAM,
    "gurmukhi":   sanscript.GURMUKHI,
    "gujarati":   sanscript.GUJARATI,
    "odia":       sanscript.ORIYA,
}

def detect_script(text: str) -> str:
    """Return dominant non-Latin script or 'latin'."""
    counts = {s: 0 for s in SCRIPT_RANGES}
    for char in text:
        cp = ord(char)
        for script, (lo, hi) in SCRIPT_RANGES.items():
            if lo <= cp <= hi:
                counts[script] += 1
    dominant = max(counts, key=counts.get)
    return dominant if counts[dominant] > 0 else "latin"

def transliterate_to_latin(text: str) -> str:
    """Convert any Indian script to Latin. Latin/French unchanged."""
    script = detect_script(text)
    if script == "latin":
        return text
    sanscript_code = SCRIPT_TO_SANSCRIPT.get(script)
    if sanscript_code is None:
        return text
    try:
        return transliterate(text, sanscript_code, sanscript.ITRANS)
    except Exception:
        return text

# ── Legal suffixes seen in real data ─────────────────────────────────────────
# Ordered longest first to avoid partial matches
LEGAL_SUFFIX_PATTERNS = [
    r"\bprivate\s+limited\b",
    r"\bpvt\.?\s*ltd\.?\b",
    r"\bl\.?l\.?c\.?\b",
    r"\bl\.?l\.?p\.?\b",
    r"\bpllc\b",
    r"\bcorporation\b",
    r"\bcorp\.?\b",
    r"\blimited\b",
    r"\bltd\.?\b",
    r"\bprivate\b",
    r"\bpvt\.?\b",
    r"\binc\.?\b",
    r"\bllp\b",
    r"\blp\b",
    # French — confirmed in output: SAS, SARL, EURL
    r"\bsarl\b",
    r"\bsasu\b",
    r"\beurl\b",
    r"\bsnc\b",
    r"\bsas\b",
    r"\bsa\b",
]

def remove_legal_suffixes(text: str) -> str:
    for pattern in LEGAL_SUFFIX_PATTERNS:
        text = re.sub(pattern, " ", text, flags=re.IGNORECASE)
    return text

# ── Address abbreviations seen in real data ───────────────────────────────────
ADDRESS_ABBREVS = {
    r"\brd\.?\b":   "road",
    r"\bst\.?\b":   "street",
    r"\bdr\.?\b":   "drive",
    r"\bave\.?\b":  "avenue",
    r"\bblvd\.?\b": "boulevard",
    r"\bapt\.?\b":  "apartment",
    r"\bfl\.?\b":   "floor",
    r"\bno\.?\b":   "number",
    r"\bhno\.?\b":  "house number",
    r"\bopp\.?\b":  "opposite",
    r"\bnr\.?\b":   "near",
    r"\bdist\.?\b": "district",
    r"\btq\.?\b":   "taluka",
    r"\bmh\b":      "maharashtra",
    r"\brj\b":      "rajasthan",
    r"\bup\b":      "uttar pradesh",
    r"\bmp\b":      "madhya pradesh",
    r"\bka\b":      "karnataka",
    r"\btn\b":      "telangana",
    r"\bap\b":      "andhra pradesh",
}

def expand_address_abbrevs(text: str) -> str:
    for pattern, replacement in ADDRESS_ABBREVS.items():
        text = re.sub(pattern, replacement, text, flags=re.IGNORECASE)
    return text

# ── Main preprocessing functions ──────────────────────────────────────────────
def preprocess_name(name: str) -> str:
    """
    Full preprocessing for business names.
    Handles all noise patterns confirmed in real data.
    """
    if not name or name.strip() == "":
        return ""

    # Step 1: Transliterate non-Latin scripts to Latin
    # Handles: Devanagari, Tamil, Telugu, Kannada, Bengali etc.
    # Also handles mixed text like "Clairvoyant Récord मध्य"
    name = transliterate_to_latin(name)

    # Step 2: Lowercase
    name = name.lower()

    # Step 3: Remove accents
    # Handles: Récord→record, Márine→marine, Société→societe
    # Also handles: CERTIFIED EMPIRE LÁTIN, WATSON JÉANNIE
    name = unicodedata.normalize("NFD", name)
    name = "".join(c for c in name if unicodedata.category(c) != "Mn")

    # Step 4: Remove leading noise characters
    # Handles: '-- Houston Bay', '... Blue', '#patriotinstitute',
    #          '@basalt', '>> HALCYNA', '# 1-QD'
    name = re.sub(r"^[\s\-\.\,\!\@\#\$\%\^\&\*\>\<\'\"\/\\]+", "", name)

    # Step 5: Remove domain extensions
    # Handles: housingnetwork.com, TEAMAIR.COM, shirdiacademy.com
    name = re.sub(r"\bwww\.", "", name, flags=re.IGNORECASE)
    name = re.sub(
        r"\.(com|net|org|co|in|fr|uk|biz|info|edu|gov|io)\b",
        "", name, flags=re.IGNORECASE
    )

    # Step 6: Remove duplicate consecutive words
    # Handles: VANNTOVA VANNTOVA, FH FH Business, PROEM PROEM
    name = re.sub(r"\b(\w+)\s+\1\b", r"\1", name, flags=re.IGNORECASE)

    # Step 7: & → and
    name = name.replace("&", " and ")

    # Step 8: Hyphen between words → space
    # Handles: Housing-Network, PLLC-PARTNERS, Lakdi-Ka
    name = re.sub(r"(?<=\w)-(?=\w)", " ", name)

    # Step 9: Remove brackets and their contents IF they wrap single words
    # Handles: Davis Family (Office)→Davis Family Office
    #          MS [Consultancy]→MS Consultancy
    #          Prairie (Capital)→Prairie Capital
    name = re.sub(r"[\(\[\{](\w+)[\)\]\}]", r"\1", name)
    # Remove remaining brackets
    name = re.sub(r"[\(\[\{\)\]\}]", " ", name)

    # Step 10: Remove legal suffixes
    # Handles: LLC, Pvt Ltd, Private Limited, SAS, SARL, EURL etc.
    name = remove_legal_suffixes(name)

    # Step 11: Remove N/A and similar placeholders
    name = re.sub(r"\bn/?a\b", "", name, flags=re.IGNORECASE)

    # Step 12: Remove remaining punctuation (keep alphanumeric and spaces)
    name = re.sub(r"[^\w\s]", " ", name)

    # Step 13: Collapse whitespace
    name = re.sub(r"\s+", " ", name).strip()

    return name


def preprocess_address(addr: str) -> str:
    """
    Full preprocessing for addresses.
    Note: Does NOT try to fix order — token_set_ratio handles that.
    Handles: Telugu/Devanagari in address, ## prefix, N/A, abbreviations.
    """
    if not addr or addr.strip() == "":
        return ""

    # Step 1: Transliterate non-Latin scripts
    # Handles: తెలంగాణ, राजस्थान, महाराष्ट्र in address field
    addr = transliterate_to_latin(addr)

    # Step 2: Lowercase
    addr = addr.lower()

    # Step 3: Remove accents
    addr = unicodedata.normalize("NFD", addr)
    addr = "".join(c for c in addr if unicodedata.category(c) != "Mn")

    # Step 4: Remove ## prefix on house numbers
    # Handles: ##19821 WHEELWRIGHT DR, ##07 19TH CROSS
    addr = re.sub(r"##\s*", "", addr)

    # Step 5: Remove N/A components
    # Handles: "CALUMET CITY, 351 HOXIE AVE, N/A, IL"
    addr = re.sub(r"\bn/?a\b", "", addr, flags=re.IGNORECASE)

    # Step 6: Expand abbreviations
    addr = expand_address_abbrevs(addr)

    # Step 7: & → and
    addr = addr.replace("&", " and ")

    # Step 8: Remove punctuation but KEEP numbers
    # Street numbers are critical matching signals
    addr = re.sub(r"[^\w\s]", " ", addr)

    # Step 9: Collapse whitespace
    addr = re.sub(r"\s+", " ", addr).strip()

    return addr

In [ ]:
# =============================================================================
# VERIFY PREPROCESSING ON REAL EXAMPLES FROM YOUR OUTPUT
# =============================================================================
print("PREPROCESSING VERIFICATION ON REAL DATA EXAMPLES")
print("=" * 65)

real_examples = [
    # (description, raw_name, raw_address)
    ("Typo in name",
     "Davis Family Offie", "88 OLIVE CIR, LEBANON, TN"),

    ("Parentheses in name",
     "Davis Family (Office)", "88 Olive Cir, Lebanon, Tennessee"),

    ("Domain name",
     "TEAMAIR.COM", "FLAT NO:101, ANUSKA TOWERS, తెలంగాణ"),

    ("Telugu in address",
     "Team Air Pvt. Ltd.", "FLAT NO:101, ANUSKA TOWERS, తెలంగాణ"),

    ("Devanagari in address",
     "FH BUSINESS PVT LIMITED", "602, 6TH FLOOR, CRYSTAL MALL, BANI PARK, JAIPUR, राजस्थान"),

    ("Devanagari name",
     "ग्रेट प्रोडक्ट्स प्रा. लि.", "NEAR SATYANARYAN TEMPLE VPO - PRAGPURA, JAIPUR, राजस्थान"),

    ("Tamil name",
     "சன் ஆனந்த் இன்ஃப்ராஸ்ட்ரக்சர் பிரைவேட் லிமிடெட்",
     "#A-4/375 OPP TO CF HOSPITAL, DINDIGUL, Tamil Nadu"),

    ("Kannada name",
     "ಅಪೆಕ್ಸ್ ಇನ್‌ಫ್ರಾ ಲಿಮಿಟೆಡ್",
     "H.NO ##07, 19TH CROSS, BANGALORE, Karnataka"),

    ("Leading punctuation",
     "-- Houston  Bay Nlf, LLC", ""),

    ("Hashtag prefix",
     "#patriotinstitute", ""),

    ("@ prefix",
     "@basalt", ""),

    (">> prefix + repeated word",
     ">> HALCYNA PROEM PROEM", "17662A CHANCELLORSVILLE AVE, LA, BATON ROUGE"),

    ("Repeated first word",
     "FH FH Business Pvt Ltd", "602, Jaipur, RJ"),

    ("Square brackets",
     "MS [Consultancy]", "SHYMALA APPTS, PUNE, Maharashtra"),

    ("Word order transposed",
     "PRAIRIE CAPITAL PLLC-PARTNERS",
     "120. Autumn Woods Boulevard, MOUNT HOLLY, NC"),

    ("French with accents",
     "Bien & Frères SAS",
     "23 Boulevard de la Renaissance, Saint-Nazaire, Pays de la Loire"),

    ("French SARL",
     "Organisme (France) Amis SARL",
     "2 Impasse Docteur Guérin, Saint-Herblain, Pays de la Loire"),

    ("## in address",
     "Drayex Berto LLC", "##19821 WHEELWRIGHT DR, MONTGOMERY VILLAGE, MD"),

    ("N/A in address",
     "Grand Connecticut LLC", "CALUMET CITY, 351 HOXIE AVE, N/A, IL"),

    ("Digit replacing letter",
     "M5  Consultancy Corp",
     "Shymala Appts, Pune, Maharashtra"),
]

for desc, raw_name, raw_addr in real_examples:
    p_name = preprocess_name(raw_name)
    p_addr = preprocess_address(raw_addr)
    print(f"\n  [{desc}]")
    print(f"  NAME  before : {raw_name}")
    print(f"  NAME  after  : {p_name}")
    if raw_addr:
        print(f"  ADDR  before : {raw_addr}")
        print(f"  ADDR  after  : {p_addr}")

# ── Similarity check on matched pairs after preprocessing ─────────────────────
print("\n\n" + "=" * 65)
print("SIMILARITY CHECK AFTER PREPROCESSING")
print("Confirm matched pairs score higher after preprocessing")
print("=" * 65)

from rapidfuzz import fuzz

matched_pair_tests = [
    ("Davis Family Office",     "Davis Family Offie"),
    ("Davis Family Office",     "DAVIS FAMILY OFFICE"),
    ("Team Air Pvt. Ltd.",      "TEAMAIR.COM"),
    ("MS Consultancy Corp",     "MS [Consultancy]"),
    ("FH Business Pvt Ltd",     "FH BUSINESS PVT LIMITED"),
    ("Grand Connecticut LLC",   "Grand Connecticut"),
    ("Prairie Capital Partners PLLC", "PRAIRIE CAPITAL PLLC-PARTNERS"),
    ("Bien & Frères SAS",       "Bien Freres"),
]

print(f"\n  {'Pair':<45} {'raw_sim':>8} {'pre_sim':>8} {'better?':>8}")
print("  " + "─" * 75)
for name1, name2 in matched_pair_tests:
    raw_sim = fuzz.token_set_ratio(name1.lower(), name2.lower())
    pre_sim = fuzz.token_set_ratio(preprocess_name(name1), preprocess_name(name2))
    better  = "✅ YES" if pre_sim >= raw_sim else "⚠️  NO"
    pair_str = f"{name1[:20]}  vs  {name2[:20]}"
    print(f"  {pair_str:<45} {raw_sim:>8} {pre_sim:>8} {better:>8}")

In [3]:
# Fix 1: n_matches column
if "n_matches" not in gt.columns:
    gt["n_matches"] = gt["matched_entity_ids"].apply(
        lambda x: len(x.split(",")) if x.strip() != "" else 0
    )

# Fix 2: lookup maps
if "s1_map" not in dir():
    s1_map = tr_s1.set_index("entity_id")[["business_name","business_address","country"]].to_dict("index")
    s2_map = tr_s2.set_index("entity_id")[["business_name","business_address","country"]].to_dict("index")
    s3_map = tr_s3.set_index("entity_id")[["business_name","business_address","country"]].to_dict("index")
    print("Lookup maps built")

Lookup maps built


In [4]:
# =============================================================================
# VERIFICATION CELL — Test blocking approach on 500 real examples
# Goal: confirm char n-gram TF-IDF + embeddings gives >85% recall
# This is NOT the full implementation — just a proof of concept
# =============================================================================

# ── Step 1: Pick 500 S1 entities from validation set that have known matches ──
sample_s1 = gt[gt["n_matches"] > 0].sample(500, random_state=42)
sample_s1_ids = set(sample_s1["source1_entity_id"])

# Get their records from train_s1
s1_sample_df = tr_s1[tr_s1["entity_id"].isin(sample_s1_ids)].copy()

# Get the S2 and S3 records of the SAME COUNTRY only
# (we always partition by country — safe and fast)
countries_in_sample = s1_sample_df["country"].unique()
s2_sample_df = tr_s2[tr_s2["country"].isin(countries_in_sample)].copy()
s3_sample_df = tr_s3[tr_s3["country"].isin(countries_in_sample)].copy()

print(f"S1 sample     : {len(s1_sample_df):,} records")
print(f"S2 pool       : {len(s2_sample_df):,} records")
print(f"S3 pool       : {len(s3_sample_df):,} records")

# Build ground truth for this sample
sample_gt = {}
for _, row in sample_s1.iterrows():
    s1_id = row["source1_entity_id"]
    matched = set(e.strip() for e in row["matched_entity_ids"].split(",") if e.strip())
    sample_gt[s1_id] = matched

total_true_matches = sum(len(v) for v in sample_gt.values())
print(f"True matches to find: {total_true_matches:,}")

S1 sample     : 500 records
S2 pool       : 5,034,616 records
S3 pool       : 5,285,603 records
True matches to find: 1,859


In [5]:
# ── Step 2: Minimal preprocessing (just enough to test) ──────────────────────
import re
import unicodedata

def quick_clean(text: str) -> str:
    """Minimal cleaning for verification — not the full pipeline."""
    if not text or text.strip() == "":
        return ""
    # Skip non-Latin — leave for embeddings
    if any(ord(c) > 0x024F for c in text):
        return text.lower().strip()
    text = text.lower()
    # Remove accents
    text = unicodedata.normalize("NFD", text)
    text = "".join(c for c in text if unicodedata.category(c) != "Mn")
    # Remove legal suffixes
    text = re.sub(r"\b(llc|inc|corp|ltd|pvt|limited|private|sas|sarl)\b", " ", text)
    # Remove punctuation
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

s1_sample_df["clean_name"] = s1_sample_df["business_name"].apply(quick_clean)
s2_sample_df["clean_name"] = s2_sample_df["business_name"].apply(quick_clean)
s3_sample_df["clean_name"] = s3_sample_df["business_name"].apply(quick_clean)

print("Preprocessing done")
print(f"Sample cleaned names:")
for name, clean in zip(s1_sample_df["business_name"].head(5),
                        s1_sample_df["clean_name"].head(5)):
    print(f"  {name!r:40} → {clean!r}")

Preprocessing done
Sample cleaned names:
  'Clairvoyant Record Private Limited'     → 'clairvoyant record'
  'Laxmi Trading Private Limited'          → 'laxmi trading'
  'Salazar Guaranty'                       → 'salazar guaranty'
  'Danit Rowland, CPA'                     → 'danit rowland cpa'
  'Family Partners LLC'                    → 'family partners'


In [7]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

True
Tesla T4


In [8]:
import torch
print("GPU available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

GPU available: True
Device: Tesla T4


In [6]:
# =============================================================================
# VERIFICATION CELL — Memory safe, tested logic
# =============================================================================
!pip install faiss-gpu

import re
import unicodedata
import numpy as np
import pandas as pd
import faiss
import gc
import torch
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
from sentence_transformers import SentenceTransformer

print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU memory: {gpu_mem:.1f} GB")

# ── Fix dependencies ──────────────────────────────────────────────────────────
if "n_matches" not in gt.columns:
    gt["n_matches"] = gt["matched_entity_ids"].apply(
        lambda x: len(x.split(",")) if x.strip() != "" else 0
    )

# ── Step 1: Sample ────────────────────────────────────────────────────────────
print("\n" + "=" * 55)
print("STEP 1: Sampling")
print("=" * 55)

sample_s1     = gt[gt["n_matches"] > 0].sample(500, random_state=42)
sample_s1_ids = set(sample_s1["source1_entity_id"])
s1_sample_df  = tr_s1[tr_s1["entity_id"].isin(sample_s1_ids)].copy()

# Country partition — only compare within same country
countries = s1_sample_df["country"].unique()
s2_sample_df = tr_s2[tr_s2["country"].isin(countries)].copy()
s3_sample_df = tr_s3[tr_s3["country"].isin(countries)].copy()

pool_df = pd.concat([
    s2_sample_df[["entity_id","business_name","business_address","country"]],
    s3_sample_df[["entity_id","business_name","business_address","country"]]
]).reset_index(drop=True)

pool_ids = pool_df["entity_id"].tolist()
s1_ids   = s1_sample_df["entity_id"].tolist()

# Ground truth
sample_gt = {}
for _, row in sample_s1.iterrows():
    s1_id = row["source1_entity_id"]
    sample_gt[s1_id] = set(
        e.strip() for e in row["matched_entity_ids"].split(",")
        if e.strip()
    )
total_true = sum(len(v) for v in sample_gt.values())

print(f"S1 sample : {len(s1_sample_df):,}")
print(f"S2 pool   : {len(s2_sample_df):,}")
print(f"S3 pool   : {len(s3_sample_df):,}")
print(f"Total pool: {len(pool_df):,}")
print(f"True matches to find: {total_true:,}")

# Memory estimate for TF-IDF
pool_size    = len(pool_df)
n_features   = 50_000
dense_gb     = pool_size * n_features * 4 / 1e9
sparse_est   = pool_size * 200 * 4 / 1e9  # ~200 non-zeros per row
print(f"\nMemory estimate:")
print(f"  Dense TF-IDF matrix would need : {dense_gb:.1f} GB  ← NEVER doing this")
print(f"  Sparse TF-IDF matrix estimate  : {sparse_est:.2f} GB  ← safe")

# ── Step 2: Preprocessing ─────────────────────────────────────────────────────
print("\n" + "=" * 55)
print("STEP 2: Preprocessing")
print("=" * 55)

def quick_clean(text: str) -> str:
    if not text or text.strip() == "":
        return ""
    if any(ord(c) > 0x024F for c in text):
        return text.strip()   # non-Latin → leave for embeddings
    text = text.lower()
    text = unicodedata.normalize("NFD", text)
    text = "".join(c for c in text if unicodedata.category(c) != "Mn")
    text = re.sub(
        r"\b(llc|inc|corp|ltd|pvt|limited|private|sas|sarl|lp|llp|pllc)\b",
        " ", text, flags=re.IGNORECASE
    )
    text = re.sub(r"\.(com|net|org|co|in|fr)\b", "", text)
    text = re.sub(r"^[\s\-\.\#\@\>]+", "", text)
    text = re.sub(r"\b(\w+)\s+\1\b", r"\1", text)
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

s1_sample_df["clean_name"] = s1_sample_df["business_name"].apply(quick_clean)
pool_df["clean_name"]       = pool_df["business_name"].apply(quick_clean)

print("Preprocessing done")
print("Sample cleaned names:")
for raw, clean in zip(
    s1_sample_df["business_name"].head(5),
    s1_sample_df["clean_name"].head(5)
):
    print(f"  {raw!r:40} → {clean!r}")

# ── Step 3: Layer 1 — Char N-gram TF-IDF (SPARSE — no .toarray() on full matrix)
print("\n" + "=" * 55)
print("STEP 3: Layer 1 — Char N-gram TF-IDF")
print("(Sparse dot product — never converts full matrix to dense)")
print("=" * 55)

vectorizer = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 4),
    min_df=3,
    max_features=50_000,
    sublinear_tf=True,
)

pool_clean = pool_df["clean_name"].fillna("").tolist()
s1_clean   = s1_sample_df["clean_name"].fillna("").tolist()

print("Fitting TF-IDF on pool (S2+S3)...")
# This stays SPARSE — never call .toarray() on this
pool_tfidf_sparse = vectorizer.fit_transform(pool_clean)
s1_tfidf_sparse   = vectorizer.transform(s1_clean)

# L2 normalize — stays sparse
pool_tfidf_sparse = normalize(pool_tfidf_sparse, norm="l2")
s1_tfidf_sparse   = normalize(s1_tfidf_sparse,   norm="l2")

print(f"Pool TF-IDF shape  : {pool_tfidf_sparse.shape}")
print(f"Pool TF-IDF memory : {pool_tfidf_sparse.data.nbytes / 1e6:.1f} MB (sparse ✅)")
print(f"S1 TF-IDF memory   : {s1_tfidf_sparse.data.nbytes / 1e6:.1f} MB (sparse ✅)")

# Chunked sparse dot product
# ONLY the result chunk (BATCH × pool_size) becomes dense — nothing else
K      = 50
BATCH  = 25   # small batch = small dense result chunk

tfidf_candidates = {}

print(f"Querying {len(s1_ids)} S1 records (batch={BATCH})...")
for i in range(0, len(s1_ids), BATCH):
    batch_ids  = s1_ids[i : i+BATCH]
    batch_vecs = s1_tfidf_sparse[i : i+BATCH]

    # Result shape: (BATCH × pool_size) — e.g. 25 × 500K = 50MB max
    # This is the ONLY dense conversion — small and immediately discarded
    sims = (batch_vecs @ pool_tfidf_sparse.T).toarray()

    top_k = np.argsort(-sims, axis=1)[:, :K]
    for j, s1_id in enumerate(batch_ids):
        tfidf_candidates[s1_id] = set(pool_ids[idx] for idx in top_k[j])

    # Free result immediately
    del sims, top_k
    if i % 100 == 0:
        gc.collect()
        print(f"  {i}/{len(s1_ids)} done")

# Recall
tfidf_found = sum(
    len(sample_gt.get(s, set()) & tfidf_candidates.get(s, set()))
    for s in sample_gt
)
tfidf_recall = tfidf_found / total_true
print(f"\nLayer 1 Recall : {tfidf_recall*100:.1f}%")
print(f"Found {tfidf_found} / {total_true}")
print(f"Target >70%  {'✅' if tfidf_recall > 0.70 else '❌'}")

# Free TF-IDF before loading embedding model
del pool_tfidf_sparse, s1_tfidf_sparse
gc.collect()
print("TF-IDF memory freed ✅")

# ── Step 4: Layer 2 — Multilingual Embeddings + FAISS ────────────────────────
print("\n" + "=" * 55)
print("STEP 4: Layer 2 — Multilingual Embeddings + FAISS")
print("(FAISS works natively with dense embeddings — no crash)")
print("=" * 55)

# Embeddings are small — 384 dims vs 50K TF-IDF dims
emb_memory_gb = len(pool_df) * 384 * 4 / 1e9
print(f"Embedding matrix size: {emb_memory_gb:.2f} GB  ← fits in RAM ✅")

model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

def make_text(name, address):
    name    = str(name)    if name    else ""
    address = str(address) if address else ""
    return f"{name} {name} {address}".strip()

pool_texts = [
    make_text(r["business_name"], r["business_address"])
    for _, r in pool_df.iterrows()
]
s1_texts = [
    make_text(r["business_name"], r["business_address"])
    for _, r in s1_sample_df.iterrows()
]

print(f"Encoding {len(pool_texts):,} pool records...")
pool_emb = model.encode(
    pool_texts,
    batch_size=512,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True,
).astype("float32")

print(f"Encoding {len(s1_texts):,} S1 records...")
s1_emb = model.encode(
    s1_texts,
    batch_size=512,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True,
).astype("float32")

print(f"Pool embedding shape : {pool_emb.shape}")
print(f"Pool embedding memory: {pool_emb.nbytes / 1e6:.0f} MB")

# FAISS index — flat inner product (safe for dense embeddings)
dim   = pool_emb.shape[1]   # 384
index = faiss.IndexFlatIP(dim)
index.add(pool_emb)
print(f"FAISS index: {index.ntotal:,} vectors")

scores, indices = index.search(s1_emb, K)

embedding_candidates = {}
for i, s1_id in enumerate(s1_ids):
    embedding_candidates[s1_id] = set(
        pool_ids[idx] for idx in indices[i] if idx != -1
    )

emb_found = sum(
    len(sample_gt.get(s, set()) & embedding_candidates.get(s, set()))
    for s in sample_gt
)
emb_recall = emb_found / total_true
print(f"\nLayer 2 Recall : {emb_recall*100:.1f}%")
print(f"Found {emb_found} / {total_true}")
print(f"Target >75%  {'✅' if emb_recall > 0.75 else '❌'}")

# ── Step 5: Combined Results ──────────────────────────────────────────────────
print("\n" + "=" * 55)
print("STEP 5: Final Verification Summary")
print("=" * 55)

combined_found = sum(
    len(
        sample_gt.get(s, set()) &
        (tfidf_candidates.get(s, set()) | embedding_candidates.get(s, set()))
    )
    for s in sample_gt
)
combined_recall = combined_found / total_true

avg_combined = sum(
    len(tfidf_candidates.get(s, set()) | embedding_candidates.get(s, set()))
    for s in s1_ids
) / len(s1_ids)

print(f"  Layer 1 TF-IDF recall    : {tfidf_recall*100:.1f}%")
print(f"  Layer 2 Embedding recall : {emb_recall*100:.1f}%")
print(f"  Combined recall (union)  : {combined_recall*100:.1f}%")
print(f"  Avg candidates per S1    : {avg_combined:.0f}")
print()
if combined_recall >= 0.85:
    print("  ✅ PLAN WORKS — safe to proceed to full implementation")
elif combined_recall >= 0.75:
    print("  ⚠️  DECENT — increase K to 100 and retest")
else:
    print("  ❌ TOO LOW — paste output here to debug")
print("=" * 55)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.3/135.3 MB 13.4 MB/s eta 0:00:0000:0100:01
GPU available: True
GPU: Tesla T4
GPU memory: 15.6 GB

STEP 1: Sampling
S1 sample : 500
S2 pool   : 5,034,616
S3 pool   : 5,285,603
Total pool: 10,320,219
True matches to find: 1,859

Memory estimate:
  Dense TF-IDF matrix would need : 2064.0 GB  ← NEVER doing this
  Sparse TF-IDF matrix estimate  : 8.26 GB  ← safe

STEP 2: Preprocessing
Preprocessing done
Sample cleaned names:
  'Clairvoyant Record Private Limited'     → 'clairvoyant record'
  'Laxmi Trading Private Limited'          → 'laxmi trading'
  'Salazar Guaranty'                       → 'salazar guaranty'
  'Danit Rowland, CPA'                     → 'danit rowland cpa'
  'Family Partners LLC'                    → 'family partners'

STEP 3: Layer 1 — Char N-gram TF-IDF
(Sparse dot product — never converts full matrix to dense)
Fitting TF-IDF on pool (S2+S3)...
Pool TF-IDF shape  : (10320219, 50000)
Pool TF-IDF memory : 2791.0 MB (sparse 

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Encoding 10,320,219 pool records...


Batches:   0%|          | 0/20157 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [7]:
# =============================================================================
# BLOCKING VERIFICATION — 4 layers, memory-safe, handles Indian scripts + typos
# =============================================================================
import re, gc, unicodedata
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import faiss
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
from sentence_transformers import SentenceTransformer

RNG_SEED   = 42
N_S1       = 1_000        # S1 entities to test
POOL_SIZE  = 1_000_000    # distractor pool size
K          = 50           # candidates per layer per S1

print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE — will be slow'}")

# ── Rebuild n_matches if missing ──────────────────────────────────────────────
if "n_matches" not in gt.columns:
    gt["n_matches"] = gt["matched_entity_ids"].apply(
        lambda x: len(x.split(",")) if x.strip() != "" else 0
    )


# =============================================================================
# STEP 1 — Build sample with guaranteed true matches in pool
# =============================================================================
print("\n" + "="*60)
print("STEP 1: Building sample")
print("="*60)

sample_s1 = gt[gt["n_matches"] > 0].sample(N_S1, random_state=RNG_SEED)

# Ground truth dict + the set of all true-match IDs we must include in the pool
sample_gt, true_match_ids = {}, set()
for _, r in sample_s1.iterrows():
    ids = {e.strip() for e in r["matched_entity_ids"].split(",") if e.strip()}
    sample_gt[r["source1_entity_id"]] = ids
    true_match_ids |= ids

total_true = sum(len(v) for v in sample_gt.values())

s1_df = tr_s1[tr_s1["entity_id"].isin(sample_gt)].reset_index(drop=True)

# Pool = all true matches + random distractors, drawn from S2 and S3
s2_true = tr_s2[tr_s2["entity_id"].isin(true_match_ids)]
s3_true = tr_s3[tr_s3["entity_id"].isin(true_match_ids)]

n_distract = POOL_SIZE - len(s2_true) - len(s3_true)
s2_rand = tr_s2[~tr_s2["entity_id"].isin(true_match_ids)].sample(
    n_distract // 2, random_state=RNG_SEED)
s3_rand = tr_s3[~tr_s3["entity_id"].isin(true_match_ids)].sample(
    n_distract - n_distract // 2, random_state=RNG_SEED)

COLS_KEEP = ["entity_id", "business_name", "business_address", "country"]
pool_df = pd.concat(
    [s2_true[COLS_KEEP], s3_true[COLS_KEEP], s2_rand[COLS_KEEP], s3_rand[COLS_KEEP]]
).reset_index(drop=True)

del s2_true, s3_true, s2_rand, s3_rand; gc.collect()

print(f"S1 entities      : {len(s1_df):,}")
print(f"True matches     : {total_true:,}")
print(f"Pool size        : {len(pool_df):,}")
print(f"Pool countries   : {pool_df['country'].value_counts().to_dict()}")

# Sanity: every true match must be present in the pool
pool_id_set = set(pool_df["entity_id"])
missing = true_match_ids - pool_id_set
assert not missing, f"BUG: {len(missing)} true matches absent from pool"
print("✅ All true matches present in pool")


# =============================================================================
# STEP 2 — Vectorized preprocessing (fast, C-level pandas ops)
# =============================================================================
print("\n" + "="*60)
print("STEP 2: Preprocessing")
print("="*60)

LEGAL = (r"\b(llc|l\.l\.c|inc|corp|corporation|ltd|limited|pvt|private|"
         r"llp|lp|pllc|sas|sarl|eurl|sasu|snc|co)\b")

def clean_series(s: pd.Series) -> pd.Series:
    """Vectorised text cleaning. Non-Latin chars are dropped (embeddings cover those)."""
    s = s.fillna("").astype(str).str.lower()
    # strip accents AND non-Latin scripts -> ascii only
    s = (s.str.normalize("NFKD")
           .str.encode("ascii", errors="ignore")
           .str.decode("ascii"))
    s = s.str.replace(r"##\s*", "", regex=True)                    # ##19821 -> 19821
    s = s.str.replace(r"\.(com|net|org|co|in|fr|uk|biz|info)\b", "", regex=True)
    s = s.str.replace(r"^[\s\-\.\,\!\@\#\$\%\^\&\*\>\<]+", "", regex=True)
    s = s.str.replace("&", " and ", regex=False)
    s = s.str.replace(LEGAL, " ", regex=True)
    s = s.str.replace(r"\bn/?a\b", " ", regex=True)
    s = s.str.replace(r"\b(\w+)\s+\1\b", r"\1", regex=True)         # FH FH -> FH
    s = s.str.replace(r"[^\w\s]", " ", regex=True)
    s = s.str.replace(r"\s+", " ", regex=True).str.strip()
    return s

ADDR_ABBR = {
    r"\brd\b": "road", r"\bst\b": "street", r"\bdr\b": "drive",
    r"\bave\b": "avenue", r"\bblvd\b": "boulevard",
    r"\bhno\b": "house number", r"\bno\b": "number",
    r"\bopp\b": "opposite", r"\bnr\b": "near",
}

def clean_addr_series(s: pd.Series) -> pd.Series:
    s = clean_series(s)
    for pat, rep in ADDR_ABBR.items():
        s = s.str.replace(pat, rep, regex=True)
    return s.str.replace(r"\s+", " ", regex=True).str.strip()

for df in (s1_df, pool_df):
    df["cname"] = clean_series(df["business_name"])
    df["caddr"] = clean_addr_series(df["business_address"])
    df["combo"] = (df["cname"] + " " + df["caddr"]).str.strip()

n_empty = (pool_df["cname"] == "").sum()
print(f"Pool rows with empty cleaned name: {n_empty:,} "
      f"({n_empty/len(pool_df)*100:.1f}%) — these are non-Latin, embeddings will catch them")
print("\nSample cleaning:")
for raw, cl in zip(s1_df['business_name'].head(4), s1_df['cname'].head(4)):
    print(f"  {raw!r:42} -> {cl!r}")


# =============================================================================
# STEP 3 — Per-country index groups
# =============================================================================
print("\n" + "="*60)
print("STEP 3: Country partitioning")
print("="*60)

countries = sorted(s1_df["country"].unique())
pool_by_country = {c: pool_df[pool_df["country"] == c].reset_index(drop=True)
                   for c in countries}
s1_by_country   = {c: s1_df[s1_df["country"] == c].reset_index(drop=True)
                   for c in countries}

for c in countries:
    print(f"  {c:<8} S1={len(s1_by_country[c]):>6,}  pool={len(pool_by_country[c]):>9,}")


# =============================================================================
# STEP 4 — Layers 1 & 2: char n-gram TF-IDF (name, and name+address)
# =============================================================================
print("\n" + "="*60)
print("STEP 4: TF-IDF layers (typo-tolerant char n-grams)")
print("="*60)

def tfidf_layer(s1_part, pool_part, text_col, k=K, batch=25):
    """Char n-gram TF-IDF blocking. Stays sparse; only a tiny batch goes dense."""
    cands = defaultdict(set)
    pool_txt = pool_part[text_col].fillna("").tolist()
    s1_txt   = s1_part[text_col].fillna("").tolist()
    pool_ids = pool_part["entity_id"].tolist()
    s1_ids   = s1_part["entity_id"].tolist()

    vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 4),
                          min_df=3, max_features=50_000, sublinear_tf=True)
    P = normalize(vec.fit_transform(pool_txt), norm="l2")
    Q = normalize(vec.transform(s1_txt),      norm="l2")
    PT = P.T.tocsc()

    for i in range(0, len(s1_ids), batch):
        sims = (Q[i:i+batch] @ PT).toarray()          # small dense chunk only
        top  = np.argpartition(-sims, min(k, sims.shape[1]-1), axis=1)[:, :k]
        for j, sid in enumerate(s1_ids[i:i+batch]):
            cands[sid] = {pool_ids[x] for x in top[j] if sims[j, x] > 0}
        del sims, top
    del P, Q, PT, vec; gc.collect()
    return cands

tfidf_name, tfidf_combo = defaultdict(set), defaultdict(set)
for c in countries:
    print(f"  [{c}] name index...", flush=True)
    for sid, v in tfidf_layer(s1_by_country[c], pool_by_country[c], "cname").items():
        tfidf_name[sid] |= v
    print(f"  [{c}] name+address index...", flush=True)
    for sid, v in tfidf_layer(s1_by_country[c], pool_by_country[c], "combo").items():
        tfidf_combo[sid] |= v
gc.collect()

def recall_of(cand_dict):
    hit = sum(len(sample_gt[s] & cand_dict.get(s, set())) for s in sample_gt)
    return hit / total_true, hit

r1, h1 = recall_of(tfidf_name)
r2, h2 = recall_of(tfidf_combo)
print(f"\n  Layer 1 (name TF-IDF)         : {r1*100:5.1f}%   {h1}/{total_true}")
print(f"  Layer 2 (name+addr TF-IDF)    : {r2*100:5.1f}%   {h2}/{total_true}")


# =============================================================================
# STEP 5 — Layer 3: multilingual embeddings + FAISS (handles all Indian scripts)
# =============================================================================
print("\n" + "="*60)
print("STEP 5: Multilingual embedding layer")
print("="*60)

model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
if torch.cuda.is_available():
    model = model.to("cuda")

def emb_text(df):
    """RAW text (not cleaned) so Devanagari/Tamil/Telugu/Kannada survive."""
    n = df["business_name"].fillna("").astype(str)
    a = df["business_address"].fillna("").astype(str)
    return (n + " " + n + " " + a).str.strip().tolist()

emb_cands = defaultdict(set)
for c in countries:
    sp, pp = s1_by_country[c], pool_by_country[c]
    print(f"  [{c}] encoding {len(pp):,} pool records...", flush=True)
    PE = model.encode(emb_text(pp), batch_size=512, normalize_embeddings=True,
                      convert_to_numpy=True, show_progress_bar=True).astype("float32")
    QE = model.encode(emb_text(sp), batch_size=512, normalize_embeddings=True,
                      convert_to_numpy=True).astype("float32")

    idx = faiss.IndexFlatIP(PE.shape[1])
    idx.add(PE)
    _, I = idx.search(QE, K)

    pids = pp["entity_id"].tolist()
    for i, sid in enumerate(sp["entity_id"].tolist()):
        emb_cands[sid] = {pids[x] for x in I[i] if x != -1}

    del PE, QE, idx, I; gc.collect()

r3, h3 = recall_of(emb_cands)
print(f"\n  Layer 3 (embeddings)          : {r3*100:5.1f}%   {h3}/{total_true}")


# =============================================================================
# STEP 6 — Layer 4: rare-token exact key match (cheap safety net)
# =============================================================================
print("\n" + "="*60)
print("STEP 6: Rare-token key layer")
print("="*60)

def token_layer(s1_part, pool_part, max_bucket=200):
    """Index pool by each name token; only rare tokens (small buckets) are used."""
    buckets = defaultdict(list)
    for eid, txt in zip(pool_part["entity_id"], pool_part["cname"]):
        for t in set(txt.split()):
            if len(t) >= 5:                       # skip short/common tokens
                buckets[t].append(eid)
    buckets = {t: v for t, v in buckets.items() if len(v) <= max_bucket}

    out = defaultdict(set)
    for eid, txt in zip(s1_part["entity_id"], s1_part["cname"]):
        for t in set(txt.split()):
            if len(t) >= 5 and t in buckets:
                out[eid].update(buckets[t])
    return out

tok_cands = defaultdict(set)
for c in countries:
    print(f"  [{c}] building token buckets...", flush=True)
    for sid, v in token_layer(s1_by_country[c], pool_by_country[c]).items():
        tok_cands[sid] |= v
gc.collect()

r4, h4 = recall_of(tok_cands)
print(f"\n  Layer 4 (rare-token keys)     : {r4*100:5.1f}%   {h4}/{total_true}")


# =============================================================================
# STEP 7 — Union and final verdict
# =============================================================================
print("\n" + "="*60)
print("FINAL RESULT")
print("="*60)

union = {s: (tfidf_name.get(s, set()) | tfidf_combo.get(s, set())
             | emb_cands.get(s, set()) | tok_cands.get(s, set()))
         for s in sample_gt}

rU, hU = recall_of(union)
avg_c  = sum(len(v) for v in union.values()) / len(union)

print(f"  Layer 1 name TF-IDF      : {r1*100:5.1f}%")
print(f"  Layer 2 name+addr TF-IDF : {r2*100:5.1f}%")
print(f"  Layer 3 embeddings       : {r3*100:5.1f}%")
print(f"  Layer 4 rare tokens      : {r4*100:5.1f}%")
print(f"  ─────────────────────────────────")
print(f"  UNION RECALL             : {rU*100:5.1f}%   ({hU}/{total_true})")
print(f"  Avg candidates per S1    : {avg_c:.0f}")
print(f"  Pool searched per S1     : ~{len(pool_df)//len(countries):,}")
print()
if rU >= 0.90:
    print("  ✅ STRONG — proceed to full implementation")
elif rU >= 0.85:
    print("  ✅ ACCEPTABLE — proceed, consider K=100 on full run")
elif rU >= 0.75:
    print("  ⚠️  RAISE K to 100 and rerun this cell")
else:
    print("  ❌ TOO LOW — paste the per-layer numbers and we debug")
print("="*60)

# ── Miss analysis: what is still being lost? ─────────────────────────────────
missed = [(s, m) for s in sample_gt
          for m in (sample_gt[s] - union.get(s, set()))]
print(f"\nMissed matches: {len(missed)}")
if missed:
    pool_lookup = pool_df.set_index("entity_id")[["business_name","business_address"]].to_dict("index")
    s1_lookup   = s1_df.set_index("entity_id")[["business_name","business_address"]].to_dict("index")
    print("First 8 misses (for diagnosis):\n")
    for s, m in missed[:8]:
        a, b = s1_lookup.get(s, {}), pool_lookup.get(m, {})
        print(f"  S1  {a.get('business_name','')}")
        print(f"      {a.get('business_address','')}")
        print(f"  MISS {b.get('business_name','')}")
        print(f"      {b.get('business_address','')}\n")

GPU: Tesla T4

STEP 1: Building sample
S1 entities      : 1,000
True matches     : 3,710
Pool size        : 1,000,000
Pool countries   : {'US': 600392, 'India': 399608}
✅ All true matches present in pool

STEP 2: Preprocessing
Pool rows with empty cleaned name: 69,004 (6.9%) — these are non-Latin, embeddings will catch them

Sample cleaning:
  'Clairvoyant Record Private Limited'       -> 'clairvoyant record'
  'Ear Nose & Throat Mountain Partners-Chicago' -> 'ear nose and throat mountain partners chicago'
  'Laxmi Trading Private Limited'            -> 'laxmi trading'
  'Anchor Payment Labs LLC'                  -> 'anchor payment labs'

STEP 3: Country partitioning
  India    S1=   392  pool=  399,608
  US       S1=   608  pool=  600,392

STEP 4: TF-IDF layers (typo-tolerant char n-grams)
  [India] name index...
  [India] name+address index...
  [US] name index...
  [US] name+address index...

  Layer 1 (name TF-IDF)         :  78.0%   2893/3710
  Layer 2 (name+addr TF-IDF)    :  97.

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  [India] encoding 399,608 pool records...


Batches:   0%|          | 0/781 [00:00<?, ?it/s]

  [US] encoding 600,392 pool records...


Batches:   0%|          | 0/1173 [00:00<?, ?it/s]


  Layer 3 (embeddings)          :  79.0%   2931/3710

STEP 6: Rare-token key layer
  [India] building token buckets...
  [US] building token buckets...

  Layer 4 (rare-token keys)     :  42.1%   1562/3710

FINAL RESULT
  Layer 1 name TF-IDF      :  78.0%
  Layer 2 name+addr TF-IDF :  97.8%
  Layer 3 embeddings       :  79.0%
  Layer 4 rare tokens      :  42.1%
  ─────────────────────────────────
  UNION RECALL             :  98.8%   (3666/3710)
  Avg candidates per S1    : 162
  Pool searched per S1     : ~500,000

  ✅ STRONG — proceed to full implementation

Missed matches: 44
First 8 misses (for diagnosis):

  S1  Best Food Private Limited
      707, Maharashtra, Level 6 & 7, Maker, Maxity No. 4 North Avenue, Mumbai
  MISS बेस्ट फूड प्राइवेट लिमिटेड
      POLT 110 707, MUMBAI, महाराष्ट्र

  S1  Jaipur Construction Pvt Ltd
      Plot No 225, Arya Nagar, Jaipur, Rajasthan
  MISS Solmiralyra
      PLOT NO 225, JAIPUR, AMBABARI, राजस्थान

  S1  Poppy Public School
      Cmc-4-1804, Gh-